# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

/bin/bash: line 1: ./.venv/bin/activate: No such file or directory


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768
SEED        = 13
EVAL_N_MCQ  = 50
EVAL_N_FREE = 50
# Local-eval tolerance estimate (DEFAULT — overridden per-question by
# infer_precision_from_question when the question text specifies precision).
# 1e-2 (1% relative) measures "the model basically got it" — generous because
# the model tends to round numerical answers to 2 decimal places. Set to 1e-8
# to recover the strict judger score, or to 1e-3 for a moderate middle ground.
EVAL_PRECISION = 1e-2
SUBSET_PATH = "results/eval_subset.json"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4.1 Preprocessing

Before sending a question to the model, we apply deterministic cleanup:

1. **LaTeX repair** — `repair_latex(text)` re-attaches missing backslashes to
   known math commands. The dataset has systematic LaTeX corruption (e.g.
   `int_{-infty}^{+infty} frac{a}{s^2+a^2}` should be
   `\int_{-\infty}^{+\infty} \frac{a}{s^2+a^2}`). Even strong external
   reasoners cannot recover from this on free-form questions; on MCQ they
   can back-infer from options at the cost of accuracy.
2. **Question preprocessing** — `preprocess_question(item)` wraps
   `repair_latex` and applies it to the question text and (if present) each
   option string. Returns a new dict; does not mutate the input.

Both functions are pure (no model dependency) and idempotent.


In [5]:
# LaTeX repair

# Common LaTeX commands appearing in the dataset's mathematical formulas.
LATEX_CMDS = [
    # Calculus / analysis
    "int", "iint", "iiint", "oint", "sum", "prod", "lim",
    "infty", "partial",
    # Fractions / roots
    "frac", "dfrac", "tfrac", "sqrt", "binom",
    # Common functions
    "sin", "cos", "tan", "cot", "sec", "csc",
    "arcsin", "arccos", "arctan",
    "sinh", "cosh", "tanh",
    "log", "ln", "exp",
    # Greek (lowercase)
    "alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta",
    "theta", "iota", "kappa", "lambda", "mu", "nu",
    "xi", "pi", "rho", "sigma", "tau", "phi", "chi", "psi", "omega",
    # Greek (uppercase)
    "Gamma", "Delta", "Theta", "Lambda", "Pi", "Sigma", "Phi", "Psi", "Omega",
    # Operators / relations
    "pm", "mp", "times", "cdot", "leq", "geq", "neq",
    "approx", "equiv", "sim", "to", "in", "notin",
    "subset", "supset", "cup", "cap",
    # Formatting
    "mathbb", "mathrm", "mathbf", "mathcal",
    "left", "right", "text",
]

# Strict context-aware pattern: only repair when the command is followed by a
# math-syntax character. This prevents false positives on standalone English
# words ("the tan of x", "in this case", "to compute") while still catching
# every real corruption in the dataset, where bare commands are always
# followed by `{`, `_`, `^`, `(`, `)`, or `}`.
_LATEX_MATH_CONTEXT = r"[{}_^()]"
_LATEX_PATTERNS = [
    (re.compile(rf"(?<!\\)\b{cmd}(?={_LATEX_MATH_CONTEXT})"), rf"\\{cmd}")
    for cmd in LATEX_CMDS
]


def repair_latex(text: str) -> str:
    """Re-attach missing backslashes to known LaTeX commands in math context.

    Only repairs when the command is immediately followed by `{`, `}`, `_`,
    `^`, `(`, or `)` (i.e. typical math syntax). This makes the function
    safe against false positives on natural English words like "tan", "in",
    "to" that collide with command names.

    Idempotent: repair_latex(repair_latex(x)) == repair_latex(x).
    """
    for pattern, repl in _LATEX_PATTERNS:
        text = pattern.sub(repl, text)
    return text


def preprocess_question(item: dict) -> dict:
    """Apply the full pre-inference preprocessing chain to a question item.

    Returns a NEW dict (does not mutate `item`). Repairs the question text
    and, if present, each option string.
    """
    out = dict(item)
    out["question"] = repair_latex(item["question"])
    if item.get("options"):
        out["options"] = [repair_latex(opt) for opt in item["options"]]
    return out

In [6]:
# Verify on real dataset samples
print("LaTeX repair on Q1 (MCQ with mangled formula)")
q1 = data[1]
print(f"  before: {q1['question']}")
print(f"  after : {repair_latex(q1['question'])}")
print(f"  options[0:3] before: {q1['options'][:3]}")
print(f"  options[0:3] after : {[repair_latex(o) for o in q1['options'][:3]]}")

print("\nFalse-positive check (English words that collide with command names)")
for s in ["the tan of x", "in this case", "to compute", "consider tangent", "Bob is tan"]:
    out = repair_latex(s)
    flag = " <= CHANGED" if out != s else ""
    print(f"  {s!r:35} -> {out!r}{flag}")

print("\nIdempotence check")
sample = "int_{-infty}^{+infty} frac{a}{s^2+a^2}"
once  = repair_latex(sample)
twice = repair_latex(once)
print(f"  repair_latex once == twice: {once == twice}")
print(f"  result: {once}")

LaTeX repair on Q1 (MCQ with mangled formula)
  before: $int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $
  after : $\int_{-\infty}^{+\infty} \frac{a^{3/2}}{s^2+a^2} ds = $
  options[0:3] before: ['$0$', '$frac{1}{a}$', '$frac{3}{a}$']
  options[0:3] after : ['$0$', '$\\frac{1}{a}$', '$\\frac{3}{a}$']

False-positive check (English words that collide with command names)
  'the tan of x'                      -> 'the tan of x'
  'in this case'                      -> 'in this case'
  'to compute'                        -> 'to compute'
  'consider tangent'                  -> 'consider tangent'
  'Bob is tan'                        -> 'Bob is tan'

Idempotence check
  repair_latex once == twice: True
  result: \int_{-\infty}^{+\infty} \frac{a}{s^2+a^2}


## 4.2 Prompt Construction

We define the system prompts and the prompt-building functions:

- `SYSTEM_PROMPT_MATH` — solve-and-box instructions for free-form questions.
- `SYSTEM_PROMPT_MCQ` — letter-only selection for multiple-choice questions.
- `MCQ_STAGE2_INSTRUCTION` — short reconciliation instruction for the two-stage
  MCQ flow (used only when stage-1's answer doesn't match any option).
- `build_prompt(question, options)` — **baseline** single-call prompt
  construction. Returns `(system, user)` where `user` includes the options
  inline for MCQ. Kept unchanged for direct comparison against v2.
- `select_prompt(item)` — **v2** conditional prompt routing. For MCQ, returns
  the *stage-1* prompts (question only, no options shown) so the model must
  derive its own answer before being anchored by the option set. For
  free-form, routes by question complexity (multi-part / long / short).
- `verify_against_options(response, options)` — uses `Judger.auto_judge` to
  check whether the stage-1 answer matches any option. Returns the matching
  letter or `None`.
- `build_mcq_stage2_messages(item, stage1_response)` — builds the multi-turn
  chat for the stage-2 reconciliation call, invoked only when
  `verify_against_options` returned `None`.

`build_prompt` is the **baseline pipeline**; `select_prompt` plus the
two-stage MCQ helpers are the **v2 pipeline**. Both coexist so we can A/B
test cleanly.


In [7]:
# generic short free-form (also reused by MCQ derivation
# stage-1 with hidden options, and as the system role in build_mcq_stage2_messages).
# Minimal per Qwen3-Thinking-2507 guidance: no verbose persona, no few-shot
# exemplars, just the official math output contract + a multi-answer safety hint.
SYSTEM_PROMPT_MATH = (
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a "
    "single \\boxed{}, e.g. \\boxed{3, 7}."
)

# MCQ with options visible in stage-1 (heuristic-flagged, single-stage).
# Used when mcq_needs_options(...) returns True. (since the model
# sees options here, charitable interpretation against the option set helps).
SYSTEM_PROMPT_MCQ = (
    "Please reason step by step, then select the single best option from the answer "
    "choices. The question may contain transcription errors (e.g. missing backslashes); "
    "interpret it charitably. Output ONLY the letter of your chosen option inside "
    "\\boxed{}, e.g. \\boxed{C}."
)

# MCQ stage-2 reconciliation user message. Fired only when stage-1
# verify_against_options returned None.
MCQ_STAGE2_INSTRUCTION = (
    "Your previous answer does not match any of the options shown above. The "
    "question may contain transcription errors (e.g. missing backslashes); "
    "reconsider charitably and choose the option that would make the problem "
    "mathematically sensible. Output ONLY the letter inside \\boxed{}, e.g. \\boxed{C}."
)

# long-context free-form (CoRe-style: restate + list conditions
# before solving). Used for free-form questions with qlen > 500. Based on
# Xu 2024 (E-GSM): forcing condition enumeration improves long-problem
# accuracy by ~5pp standalone, ~15pp with self-consistency. No exemplars
# (degrades Thinking models per DeepSeek-R1 / Qwen3 guidance).
SYSTEM_PROMPT_LONG = (
    "Before solving, restate the problem in your own words and list every given "
    "condition and the quantity to find. Then solve.\n\n"
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a "
    "single \\boxed{}, e.g. \\boxed{3, 7}."
)

# multi-part free-form. Used when the question has more than one
# [ANS] placeholder. The dataset has 415 such questions (55% of free-form),
# of which 178 are also long (>500 chars). Failure mode is format
# compliance, not reasoning depth — model needs to (a) produce the right
# COUNT of answers in the right ORDER, (b) put them all inside ONE \boxed,
# (c) not interleave reasoning between answers (the judger only keeps the
# last contiguous group of \boxed expressions).
SYSTEM_PROMPT_MULTIPART = (
    "This problem contains multiple sub-answers marked by [ANS] placeholders "
    "in the question. Identify each sub-question, then solve them in the order "
    "they appear.\n\n"
    "After completing all reasoning, output every answer in a SINGLE \\boxed{}, "
    "comma-separated, in the same order as the [ANS] slots — for example "
    "\\boxed{a, b, c}. Do not split answers across multiple \\boxed{} expressions."
)


# Heuristic: does this MCQ require options to be visible in the prompt?
# Some MCQs ask "which of the following ... is true / unbiased / valid" — the
# question is *about* the option set itself; deriving without options is
# incoherent. For those, route to a single-stage prompt with options inline
# (skip verify + stage-2). False positives just fall back to baseline behavior;
# false negatives are still recovered by the stage-2 reconciliation call.
_OPTIONS_REFERENCED = re.compile(
    r"\b(?:"
    r"which (?:of the )?(?:following|options?|statements?|choices?)"
    r"|(?:all|none) of the above"
    r"|select all"
    r"|what (?:is|are) (?:correct|right|true)"
    r"|determine the (?:corresponding )?output"
    r")\b",
    re.IGNORECASE,
)
_OPT_SELF_REFERENCE = re.compile(r"\b(?:all|none) of the above\b", re.IGNORECASE)


def mcq_needs_options(question: str, options: list) -> bool:
    """True iff the question text or option set requires options to be visible."""
    if _OPTIONS_REFERENCED.search(question):
        return True
    return any(_OPT_SELF_REFERENCE.search(opt) for opt in options)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Baseline single-call prompt construction (kept for direct comparison).

    Returns (system_prompt, user_prompt) where user_prompt includes the
    options inline for MCQ items. Used by the original/baseline pipeline.
    """
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


def select_prompt(item: dict) -> tuple[str, str, bool]:
    """Conditional prompt routing: return (system, user, options_visible).

    The third return value is the single source of truth for "this item saw
    options in stage-1, so skip verify and stage-2." Routing rules:
      - Free-form, multi-part ([ANS] > 1)      => SYSTEM_PROMPT_MULTIPART (slot 2)
      - Free-form, long (qlen > 500)           => SYSTEM_PROMPT_LONG (slot 3, CoRe-style)
      - Free-form, short                       => SYSTEM_PROMPT_MATH (slot 1)
      - MCQ flagged by mcq_needs_options       => SYSTEM_PROMPT_MCQ + options inline (single-stage)
      - MCQ otherwise (pure derivation)        => SYSTEM_PROMPT_MATH, no options (slot 4, two-stage flow)
    """
    question = item["question"]
    options  = item.get("options")

    if not options:
        # Free-form: route by complexity
        n_ans = question.count("[ANS]")
        qlen  = len(question)
        if n_ans > 1:
            system = SYSTEM_PROMPT_MULTIPART   # slot 2 — multi-part format enforcement
        elif qlen > 500:
            system = SYSTEM_PROMPT_LONG        # slot 3 — CoRe-style restate+enumerate
        else:
            system = SYSTEM_PROMPT_MATH        # slot 1 — generic short
        return system, question, False

    if mcq_needs_options(question, options):
        # MCQ that references its options — show them in stage-1
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}", True

    # MCQ derivation problem — slot 4, hide options for stage-1, reuse slot 1 prompt
    return SYSTEM_PROMPT_MATH, question, False


def verify_against_options(response: str, options: list) -> Optional[str]:
    """Check if the model's answer matches any option. Short-circuits on first match.

    Returns the matching letter ('A', 'B', ...) or None if no option matched.
    The caller uses None as the signal to trigger the stage-2 reconciliation call.
    """
    sys.path.insert(0, ".")
    from judger import Judger
    judger_local = Judger(strict_extract=True)

    for i, opt in enumerate(options):
        try: # treat every option as the answer, if the response match any of them, matched
            if judger_local.auto_judge(pred=response, gold=[opt], options=[[]]):
                return chr(65 + i) # chacater start from A (65)
        except Exception:
            continue
    return None

def build_mcq_stage2_messages(item: dict, stage1_response: str) -> list:
    """Build chat messages for the stage-2 MCQ reconciliation call.

    Multi-turn conversation: stage-1 question (no options) => stage-1
    response => user shows options and asks to reconsider. The chat history
    gives the model context that it already attempted the problem.
    """
    question = item["question"]
    options  = item["options"]
    labels   = [chr(65 + i) for i in range(len(options))]
    opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))

    return [
        {"role": "system",    "content": SYSTEM_PROMPT_MATH},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": stage1_response},
        {"role": "user",      "content": f"Options:\n{opts_text}\n\n{MCQ_STAGE2_INSTRUCTION}"},
    ]



def verify_against_options_lenient(
    response: str,
    options: list,
    precision: float = 1e-2,
) -> Optional[str]:
    """Lenient option matcher — does NOT use Judger.auto_judge.

    Picks the BEST match across all options (not first-match-wins).
    Strategies tried per option:
      1. Exact cleaned string match            -> immediate win.
      2. Numeric comparison: relative distance  -> track minimum distance.
      3. Sympy SYMBOLIC equality only           -> counts as distance 0.
      4. Decimal-compat at common decimal places -> counts as distance 0.

    Internal precision is clamped to <=1e-4. MCQ distractors are designed
    to be close (often within 1% of each other); the caller's precision is
    calibrated for free-form scoring and would over-match across multiple
    options. The numeric / decimal-compat paths require BOTH sides to be
    clean single numbers — options like "0 or -3" or "x + 1" can still hit
    the exact-string or sympy-equality paths but won't get spuriously
    matched by substring numerics.
    """
    sys.path.insert(0, ".")
    from judger import Judger
    pred = Judger(strict_extract=True).extract_ans(response)
    if not pred:
        return None

    # MCQ-internal precision floor — see docstring.
    precision = min(precision, 1e-4)

    def _clean(s: str) -> str:
        s = s.strip().strip("$").strip()
        # Strip trailing punctuation that often appears in MCQ option text.
        # E.g. "-3;" vs model's "-3" should match. Don't strip from the
        # middle — only the tail — so "0 or -3;" doesn't lose its semantic
        # structure.
        return s.rstrip(";,.!? ").strip()

    _CLEAN_NUM_RE = re.compile(r"^-?(?:\d+(?:\.\d+)?|\.\d+|\d+/\d+)$")
    def _is_clean_number(s: str) -> bool:
        """True iff s is a single number literal — no 'or', no text, no expr."""
        return bool(_CLEAN_NUM_RE.match(s.strip()))

    def _to_float(s: str) -> float:
        s = s.strip().replace(",", "")
        if "/" in s:
            num, den = s.split("/", 1)
            return float(num) / float(den)
        return float(s)

    def _numeric_distance(a: str, b: str) -> Optional[float]:
        """Relative distance |a-b|/|b| if both clean numbers; else None."""
        if not (_is_clean_number(a) and _is_clean_number(b)):
            return None
        try:
            av, bv = _to_float(a), _to_float(b)
            denom = max(abs(bv), 1e-12)
            return abs(av - bv) / denom
        except (ValueError, TypeError, ZeroDivisionError):
            return None

    def _try_sympy_equality(a: str, b: str) -> bool:
        """EXACT symbolic equality. No .evalf() fallback."""
        try:
            from sympy import simplify, sympify
            from sympy.parsing.latex import parse_latex
        except ImportError:
            return False

        def _parse(s: str):
            try:
                return parse_latex(s)
            except Exception:
                pass
            s2 = s.replace("\\cdot", "*").replace("\\times", "*").replace("^", "**")
            try:
                return sympify(s2)
            except Exception:
                return None

        ae, be = _parse(a), _parse(b)
        if ae is None or be is None:
            return False
        try:
            return simplify(ae - be) == 0
        except Exception:
            return False

    def _try_decimal_compat(a: str, b: str) -> bool:
        """Round both to the common decimal-place count; equal?"""
        if not (_is_clean_number(a) and _is_clean_number(b)):
            return False
        try:
            av = _to_float(a)
            bv = _to_float(b)
        except (ValueError, TypeError, ZeroDivisionError):
            return False

        def _dp_count(s: str) -> int:
            s_clean = s.strip().replace(",", "").lstrip("-+")
            if "." not in s_clean:
                return 0
            return len(s_clean.split(".")[1].rstrip("0"))

        a_dp, b_dp = _dp_count(a), _dp_count(b)
        common_dp = min(a_dp, b_dp)
        if common_dp == 0:
            return abs(av - bv) < 0.5
        return round(av, common_dp) == round(bv, common_dp)

    pred_norm = _clean(pred)

    # Score every option; keep best (lowest numeric distance; 0 for symbolic/decimal matches).
    best_letter   = None
    best_distance = float("inf")

    for i, opt in enumerate(options):
        letter   = chr(65 + i)
        opt_norm = _clean(opt)

        # 1. Exact cleaned-string match — immediate win.
        if pred_norm == opt_norm:
            return letter

        # 2. Numeric — track minimum relative distance.
        d = _numeric_distance(pred_norm, opt_norm)
        if d is not None:
            if d < precision and d < best_distance:
                best_distance = d
                best_letter   = letter
            continue  # numeric path was applicable; don't double-count via sympy/decimal

        # 3. Sympy symbolic equality — counts as a distance-0 candidate.
        if _try_sympy_equality(pred_norm, opt_norm):
            if 0 < best_distance:
                best_distance = 0
                best_letter   = letter
            continue

        # 4. Decimal-compat — distance-0 candidate.
        if _try_decimal_compat(pred_norm, opt_norm):
            if 0 < best_distance:
                best_distance = 0
                best_letter   = letter

    return best_letter


_WORD2DIGIT = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
}
_NUM_TOKEN = r"(\d+|" + "|".join(_WORD2DIGIT) + r")"


def _parse_num_token(tok: str) -> int:
    return int(tok) if tok.isdigit() else _WORD2DIGIT[tok.lower()]


def infer_precision_from_question(question: str, default: float = 1e-2) -> float:
    """Parse question text for explicit precision spec; return relative tolerance.
    Mappings: "exact" → 1e-8; "N decimal places" → 10^-N; "N sig figs" →
    10^-(N-1); "nearest tenth/hundredth/..." → 1e-1/1e-2/... Otherwise default.
    """
    q = (question or "").lower()
    if re.search(r"\b(?:exact(?:\s+(?:value|answer))?|do not round|no rounding)\b", q):
        return 1e-8
    m = re.search(
        rf"(?:to|correct to|accurate to|round(?:ed)? to|at least|use(?:s|d|ing)?|with)\s+{_NUM_TOKEN}\s+decimal\s+places?",
        q,
    )
    if m:
        return 10 ** (-_parse_num_token(m.group(1)))
    m = re.search(
        rf"(?:to|correct to|accurate to|with|using)\s+{_NUM_TOKEN}\s+(?:significant\s+(?:figures?|digits?)|sig\.?\s*figs?)",
        q,
    )
    if m:
        return 10 ** (-(_parse_num_token(m.group(1)) - 1))
    m = re.search(r"nearest\s+(tenth|hundredth|thousandth|ten-thousandth|hundred-thousandth)", q)
    if m:
        return {"tenth":1e-1,"hundredth":1e-2,"thousandth":1e-3,"ten-thousandth":1e-4,"hundred-thousandth":1e-5}[m.group(1)]
    return default

In [8]:
# Verify on real dataset samples
print("Baseline build_prompt on MCQ sample")
sys_p, usr_p = build_prompt(mcq_sample["question"], mcq_sample.get("options"))
print(f"  system: {sys_p[:60]}...")
print(f"  user  : {usr_p[:100]}... (includes options)")

print("\nselect_prompt on MCQ sample (derivation MCQ — stage 1, no options)")
prepped = preprocess_question(mcq_sample)
sys_p, usr_p, opts_visible = select_prompt(prepped)
print(f"  system: {sys_p[:60]}...")
print(f"  user  : {usr_p[:80]}...")
print(f"  options_visible = {opts_visible}  (False => two-stage flow)")

print("\nRouting on free-form sample (Q0, short)")
prepped = preprocess_question(free_sample)
sys_p, usr_p, opts_visible = select_prompt(prepped)
print(f"  qlen={len(prepped['question'])} n_ans={prepped['question'].count('[ANS]')}")
print(f"  system: {sys_p[:60]}...")
print(f"  options_visible = {opts_visible}")

print("\nHeuristic check on an options-required MCQ (id 281, 'which of the following ... is unbiased')")
opt_required_sample = next((d for d in data if d.get("id") == 281), None)
if opt_required_sample is not None:
    prepped = preprocess_question(opt_required_sample)
    sys_p, usr_p, opts_visible = select_prompt(prepped)
    print(f"  system: {sys_p[:60]}...")
    print(f"  user  : {usr_p[:120]}...")
    print(f"  options_visible = {opts_visible}  (True => single-stage with options inline)")
else:
    print("  (id 281 not present in this dataset)")

print("\nStage-2 message structure (mock — no real stage-1 response yet)")
mock_stage1 = "After computing the integral, I get \\boxed{2/a}."
msgs = build_mcq_stage2_messages(preprocess_question(mcq_sample), mock_stage1)
for m in msgs:
    print(f"  [{m['role']:9s}] {m['content'][:80]}...")


Baseline build_prompt on MCQ sample
  system: Please reason step by step, then select the single best opti...
  user  : $int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}... (includes options)

select_prompt on MCQ sample (derivation MCQ — stage 1, no options)
  system: Please reason step by step, and put your final answer within...
  user  : $\int_{-\infty}^{+\infty} \frac{a^{3/2}}{s^2+a^2} ds = $...
  options_visible = False  (False => two-stage flow)

Routing on free-form sample (Q0, short)
  qlen=71 n_ans=1
  system: Please reason step by step, and put your final answer within...
  options_visible = False

Heuristic check on an options-required MCQ (id 281, 'which of the following ... is unbiased')
  system: Please reason step by step, then select the single best opti...
  user  : Let the population (X) be uniformly distributed on ([0, \theta]). A sample of size 1, (X_1), is drawn from this populati...
  options_visible = True  (Tru

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90,
    max_model_len=49152,
    trust_remote_code=True,
    max_num_seqs=64,
    max_num_batched_tokens=16384,
    enforce_eager=False,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    seed=SEED,
)

print("Model loaded.")


INFO 05-28 21:45:18 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 49152, 'enable_prefix_caching': False, 'max_num_batched_tokens': 16384, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-28 21:45:19 [model.py:549] Resolved architecture: Qwen3ForCausalLM


INFO 05-28 21:45:19 [model.py:1678] Using max model len 49152


INFO 05-28 21:45:19 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.


INFO 05-28 21:45:20 [vllm.py:790] Asynchronous scheduling is enabled.


...

Capturing CUDA graphs (decode, FULL):  27%|██▋       | 3/11 [00:00<00:00,  9.98it/s]

Capturing CUDA graphs (decode, FULL):  64%|██████▎   | 7/11 [00:00<00:00, 10.18it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 11/11 [00:01<00:00, 10.51it/s]


(EngineCore pid=3063) INFO 05-28 21:45:50 [gpu_model_runner.py:6046] Graph capturing finished in 4 secs, took 1.95 GiB
(EngineCore pid=3063) INFO 05-28 21:45:50 [gpu_worker.py:597] CUDA graph pool memory: 1.95 GiB (actual), 2.29 GiB (estimated), difference: 0.34 GiB (17.4%).
(EngineCore pid=3063) INFO 05-28 21:45:50 [core.py:283] init engine (profile, create kv cache, warmup model) took 12.85 seconds


(EngineCore pid=3063) INFO 05-28 21:45:52 [vllm.py:790] Asynchronous scheduling is enabled.
Model loaded.


In [10]:
# # from new starter code --- will be useful once we enter SFT phase
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [11]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


In [12]:
import hashlib

# Build/Load the stratified eval subset ids
if Path(SUBSET_PATH).exists():
    with open(SUBSET_PATH) as f:
        subset_ids = json.load(f)
    print(f"Loaded existing subset of {len(subset_ids)} IDs from {SUBSET_PATH}")
else:
    import random
    rng = random.Random(SEED)

    def stratify_by_length(items, n):
        items = sorted(items, key=lambda d: len(d["question"]))
        b = len(items) // 3
        buckets = [items[:b], items[b:2*b], items[2*b:]] # split into 3 pools, each represent the specifc length portion
        base, rem = divmod(n, 3) # pick base number of questions from each pool
        chosen = []
        for i, bucket in enumerate(buckets):
            k = base + (1 if i < rem else 0)
            chosen.extend(rng.sample(bucket, k))
        return chosen

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]
    chosen = stratify_by_length(mcq_pool, EVAL_N_MCQ) + stratify_by_length(free_pool, EVAL_N_FREE)
    subset_ids = [d["id"] for d in chosen]

    Path(SUBSET_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(SUBSET_PATH, "w") as f:
        json.dump(subset_ids, f, indent=2)
    print(f"Saved new subset of {len(subset_ids)} IDs to {SUBSET_PATH}")

Loaded existing subset of 100 IDs from results/eval_subset.json


In [13]:
# Load eval subset with ids
id_set = set(subset_ids)
subset = [d for d in data if d["id"] in id_set]
n_mcq  = sum(1 for d in subset if d.get("options"))
print(f"Eval subset: {len(subset)} questions ({n_mcq} MCQ, {len(subset)-n_mcq} free-form)")

Eval subset: 100 questions (50 MCQ, 50 free-form)


For save/resume from interuption

In [14]:
# Cache key: include all v2 prompts + pipeline tag so v1 cache survives
PIPELINE_TAG = "v2.8"
prompt_hash = hashlib.md5(
    (SYSTEM_PROMPT_MATH + "||" + SYSTEM_PROMPT_MCQ + "||"
     + MCQ_STAGE2_INSTRUCTION + "||" + SYSTEM_PROMPT_LONG + "||"
     + SYSTEM_PROMPT_MULTIPART + "||" + PIPELINE_TAG).encode()
).hexdigest()[:8]
CACHE_PATH = f"results/cache/{prompt_hash}_seed{SEED}.jsonl"
Path(CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)

# Cache loader: only read 'response' (scorer needs); ignore 'stage1' (diagnostic)
cache = {}
if Path(CACHE_PATH).exists():
    with open(CACHE_PATH) as f:
        for line in f:
            e = json.loads(line)
            cache[e["id"]] = e["response"]
print(f"Cache hits: {len(cache)} for prompt hash {prompt_hash} ({PIPELINE_TAG})")

to_generate = [item for item in subset if item["id"] not in cache]
n_to_gen_mcq  = sum(1 for it in to_generate if it.get("options"))
print(f"Need to generate: {len(to_generate)} ({n_to_gen_mcq} MCQ stage-1, {len(to_generate)-n_to_gen_mcq} free-form)")


Cache hits: 70 for prompt hash 5055b013 (v2.7)
Need to generate: 30 (14 MCQ stage-1, 16 free-form)


In [15]:
# v2.1 mini-batched generation with heuristic-routed MCQ flow (crash-safe)
# Per batch:
# 1. Stage-1: generate for ALL items
#    - Free-form: solve normally (stage-1 IS the answer)
#    - MCQ where options are referenced: prompt includes options inline (single-stage)
#    - MCQ derivation: prompt hides options (two-stage flow)
# 2. Verify MCQ stage-1 answers (only for hidden-options MCQs)
# 3. Stage-2: generate ONLY for hidden-options MCQs where verify failed
# 4. Append batch results to cache file (saves on crash)
BATCH_SIZE = 10

total_batches = (len(to_generate) + BATCH_SIZE - 1) // BATCH_SIZE if to_generate else 0
total_mcq_visible_opts = 0
total_mcq_s1_hits = 0
total_stage2_calls = 0

for batch_start in range(0, len(to_generate), BATCH_SIZE):
    batch = to_generate[batch_start:batch_start + BATCH_SIZE]
    batch_num = batch_start // BATCH_SIZE + 1
    prepped = [preprocess_question(item) for item in batch]

    # Stage 1: build prompts and generate for all items in batch
    stage1_prompts = []
    options_visible_flags = []
    for p in prepped:
        sys_p, user_p, opts_visible = select_prompt(p)
        options_visible_flags.append(opts_visible)
        # formulate the prompt to the exact string the model expects
        stage1_prompts.append(tokenizer.apply_chat_template(
            [{"role": "system", "content": sys_p},
             {"role": "user",   "content": user_p}],
            tokenize=False,
            add_generation_prompt=True,
        ))
    stage1_outs = llm.generate(stage1_prompts, sampling_params=sampling_params)
    stage1_texts = [o.outputs[0].text.strip() for o in stage1_outs]

    # Classify each item: free-form / MCQ visible-opts / MCQ s1-hit / MCQ needs stage-2
    final_response = [None] * len(batch)
    stage1_field   = [None] * len(batch)   # diagnostic: only set for hidden-opts MCQs
    needs_stage2_idx = []
    n_mcq_visible_opts = 0
    n_mcq_s1_hits = 0
    n_freeform = 0

    for i, (orig_item, prep_item, s1) in enumerate(zip(batch, prepped, stage1_texts)):
        if not orig_item.get("options"):
            # Free-form: stage-1 IS the final answer
            final_response[i] = s1
            n_freeform += 1
        elif options_visible_flags[i]:
            # MCQ with options visible in stage-1: trust the model's letter, no verify
            final_response[i] = s1
            n_mcq_visible_opts += 1
        else:
            # MCQ derivation: try to match stage-1 answer against options.
            # Lenient verify (sympy + decimal_compat fallbacks) catches the
            # judger-strict cases documented in ProblemsNQuesions.md.
            stage1_field[i] = s1
            item_precision = infer_precision_from_question(
                prep_item["question"], default=EVAL_PRECISION
            )
            letter = verify_against_options_lenient(
                s1, prep_item["options"], precision=item_precision
            )
            if letter is not None:
                # Synthetic boxed letter — clean, unambiguous for the scorer
                final_response[i] = f"\\boxed{{{letter}}}"
                n_mcq_s1_hits += 1
            else:
                needs_stage2_idx.append(i)

    # Stage 2: only call llm.generate if there is real work
    if needs_stage2_idx:
        s2_prompts = []
        for i in needs_stage2_idx:
            msgs = build_mcq_stage2_messages(prepped[i], stage1_texts[i])
            s2_prompts.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            ))
        s2_outs = llm.generate(s2_prompts, sampling_params=sampling_params)
        for k, i in enumerate(needs_stage2_idx):
            final_response[i] = s2_outs[k].outputs[0].text.strip()

    # Append-save the batch (crash-safe: if killed mid-run, prior batches survive)
    with open(CACHE_PATH, "a") as f:
        for i, item in enumerate(batch):
            rec = {"id": item["id"], "response": final_response[i]}
            if stage1_field[i] is not None:
                rec["stage1"] = stage1_field[i]
            cache[item["id"]] = final_response[i]
            f.write(json.dumps(rec) + "\n")

    total_mcq_visible_opts += n_mcq_visible_opts
    total_mcq_s1_hits += n_mcq_s1_hits
    total_stage2_calls += len(needs_stage2_idx)
    print(f"Batch {batch_num}/{total_batches}: "
          f"MCQ visible-opts={n_mcq_visible_opts}, hidden s1-hit={n_mcq_s1_hits}, "
          f"stage-2={len(needs_stage2_idx)}, free-form={n_freeform} "
          f"| total cached={len(cache)}/{len(subset)}")

if to_generate:
    print(f"\nGeneration complete. MCQ visible-opts: {total_mcq_visible_opts}, "
          f"hidden s1-hits: {total_mcq_s1_hits}, stage-2 calls: {total_stage2_calls}")

# Align responses with subset order
responses = [cache[item["id"]] for item in subset]
print(f"Ready: {len(responses)} responses for scoring")

# Preview first 2 responses
for i in range(min(2, len(responses))):
    print(f"\n── Response (id={subset[i]['id']}, type={'MCQ' if subset[i].get('options') else 'free-form'}) ──")
    print(responses[i][:300], "..." if len(responses[i]) > 300 else "")


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 10/10 [00:00<00:00, 407.23it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:  10%|█         | 1/10 [00:18<02:43, 18.20s/it, est. speed input: 8.02 toks/s, output: 30.93 toks/s]

Processed prompts:  20%|██        | 2/10 [00:24<01:27, 10.99s/it, est. speed input: 10.03 toks/s, output: 54.15 toks/s]

...

Processed prompts:  50%|█████     | 1/2 [01:41<01:41, 101.17s/it, est. speed input: 18.34 toks/s, output: 31.13 toks/s]

Processed prompts: 100%|██████████| 2/2 [02:19<00:00, 64.03s/it, est. speed input: 27.91 toks/s, output: 71.01 toks/s] 

Processed prompts: 100%|██████████| 2/2 [02:19<00:00, 64.03s/it, est. speed input: 27.91 toks/s, output: 71.01 toks/s]

Processed prompts: 100%|██████████| 2/2 [02:19<00:00, 69.60s/it, est. speed input: 27.91 toks/s, output: 71.01 toks/s]

Batch 3/3: MCQ visible-opts=0, hidden s1-hit=2, stage-2=2, free-form=6 | total cached=100/100

Generation complete. MCQ visible-opts: 0, hidden s1-hits: 5, stage-2 calls: 9
Ready: 100 responses for scoring

── Response (id=20, type=free-form) ──
Okay, let's tackle this problem step by step. First, I need to find the standard deviation of the data set: 22, 23, 33, 35, 39, 40. The problem gives a table with some [ANS] placeholders, so I need to fill in each of those. Let me recall that standard deviation (s) for a sample is calculated using t ...

── Response (id=42, type=MCQ) ──
\boxed{A} 


In [16]:
# Smoke test: lightweight debug checks before scoring
# Catches obvious problems (length mismatch, missing boxed letters) without
# running the full Judger pipeline.

# 1. responses aligned with subset
assert len(responses) == len(subset), \
    f"Length mismatch: {len(responses)} responses vs {len(subset)} subset items"

# 2. every MCQ has a single-letter \boxed{} somewhere in its response
mcq_missing_box = []
for item, resp in zip(subset, responses):
    if item.get("options") and not re.search(r"\\boxed\{[A-Z]\}", resp):
        mcq_missing_box.append(item["id"])
if mcq_missing_box:
    print(f"WARNING: {len(mcq_missing_box)} MCQ items have no single-letter \\boxed: {mcq_missing_box}")
    print("  (scorer will fall back to last standalone capital letter — acceptable but worth noting)")
else:
    print(f"OK: all {sum(1 for d in subset if d.get('options'))} MCQ items have a \\boxed{{LETTER}}")

# 3. Show which MCQ ids took the visible-options route (heuristic-flagged)
visible_opt_ids = [
    item["id"] for item in subset
    if item.get("options") and mcq_needs_options(item["question"], item["options"])
]
print(f"\nMCQ items routed visible-options (n={len(visible_opt_ids)}): {visible_opt_ids}")

# 4. Eyeball one stage-1-hit and one stage-2-used MCQ entry from the cache
print("\nSample cache entries (for human inspection):")
sample_count = 0
with open(CACHE_PATH) as f:
    for line in f:
        if sample_count >= 2:
            break
        e = json.loads(line)
        if "stage1" not in e:
            continue
        is_synthetic = e["response"].startswith("\\boxed{") and len(e["response"]) <= 12
        kind = "stage-1 hit (synthetic \\boxed{})" if is_synthetic else "stage-2 used (full response)"
        resp_preview   = e["response"][:120] + ("..." if len(e["response"]) > 120 else "")
        stage1_preview = e["stage1"][:120]   + ("..." if len(e["stage1"])   > 120 else "")
        print(f"\n  [id={e['id']}] {kind}")
        print(f"    response: {resp_preview}")
        print(f"    stage1  : {stage1_preview}")
        sample_count += 1


OK: all 50 MCQ items have a \boxed{LETTER}

MCQ items routed visible-options (n=5): [306, 313, 316, 541, 636]

Sample cache entries (for human inspection):

  [id=42] stage-1 hit (synthetic \boxed{})
    response: \boxed{A}
    stage1  : This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think a...

  [id=81] stage-2 used (full response)
    response: Okay, let's go through this again carefully. The problem states that the rank of the vector group is 2, and we need to f...
    stage1  : Okay, let's try to figure out this problem. So we have three vectors alpha1, alpha2, alpha3, each is a column vector wit...


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [17]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(subset, responses), total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        # Per-question precision (e.g. "to 3 decimal places" → 1e-3, "exact" → 1e-8);
        # falls back to EVAL_PRECISION when the question doesn't specify.
        item_precision = infer_precision_from_question(
            item["question"], default=EVAL_PRECISION
        )
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
                precision=item_precision,
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/100 [00:00<?, ?it/s]

Scoring:   1%|          | 1/100 [00:00<00:18,  5.46it/s]

Scoring:  12%|█▏        | 12/100 [00:00<00:01, 47.53it/s]

Scoring:  18%|█▊        | 18/100 [00:00<00:02, 39.06it/s]

Scoring:  24%|██▍       | 24/100 [00:00<00:01, 44.03it/s]

...

Scoring:  76%|███████▌  | 76/100 [00:00<00:00, 104.95it/s]

Scoring:  88%|████████▊ | 88/100 [00:01<00:00, 38.91it/s] 

Scoring:  97%|█████████▋| 97/100 [00:02<00:00, 32.96it/s]

Scoring: 100%|██████████| 100/100 [00:02<00:00, 45.10it/s]

Scoring complete. 100 results.


## 8. Summary

Print accuracy broken down by question type.

In [18]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

# Per-length-bucket accuracy (within stratified subset)
sorted_q = sorted(subset, key=lambda d: len(d["question"]))
bsize = len(sorted_q) // 3
length_buckets = {
    "short ": [d["id"] for d in sorted_q[:bsize]],
    "medium": [d["id"] for d in sorted_q[bsize:2*bsize]],
    "long  ": [d["id"] for d in sorted_q[2*bsize:]],
}
print("\nBy question length:")
for label, ids in length_buckets.items():
    s = [r for r in results if r["id"] in set(ids)]
    if s:
        print(f"  {label}: {sum(r['correct'] for r in s):2d} / {len(s):2d}  ({acc(s):.2f}%)")

# Per-routing-path accuracy (MCQ only) — diagnoses whether the heuristic
# is paying off vs the two-stage hidden-options flow.
print("\nBy routing path (MCQ only):")
for label, pred in [
    ("visible-opts", lambda it: mcq_needs_options(it["question"], it["options"])),
    ("hidden-opts ", lambda it: not mcq_needs_options(it["question"], it["options"])),
]:
    ids = {it["id"] for it in subset if it.get("options") and pred(it)}
    s = [r for r in results if r["id"] in ids]
    if s:
        print(f"  {label}: {sum(r['correct'] for r in s):2d} / {len(s):2d}  ({acc(s):.2f}%)")


EVALUATION RESULTS
  MCQ        :   35 /   50  (70.00%)
  Free-form  :   43 /   50  (86.00%)
  Overall    :   78 /  100  (78.00%)

By question length:
  short : 26 / 33  (78.79%)
  medium: 27 / 33  (81.82%)
  long  : 25 / 34  (73.53%)

By routing path (MCQ only):
  visible-opts:  4 /  5  (80.00%)
  hidden-opts : 31 / 45  (68.89%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [19]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!